# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **Which neighborhoods have both high crime and high blight?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-04-21 15:33:00 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-04-21T15:33:00.899051")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Search for Datasets

**Search query:** `crime incidents`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 6 datasets matching 'crime incidents'

1. **Police Incident Blotter (Archived)**
   ID: `uniform-crime-reporting-data`
   **This data ceased updating with the transition to a new records management system on 11/14/2023. Access to the updated data set has been added as of April 11, 2025 here: <a href="
   - Blotter Data (UCR Coded) (CSV) [DataStore] ID: `044f2016-1dfd-4ab0-bc1e-065da05fca2e`
   - Archived Police Incident Blotter Data Dictionary (XLSX) [DataStore] ID: `a0e233b3-8cfc-441a-a37e-d396579d20ea`
   - Historical Blotter Data (CSV) [DataStore] ID: `391942e2-25ef-43e4-8263-f8519fa8
```


In [ ]:
# Step 1: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'crime incidents', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Search for Datasets

**Search query:** `blight property violations`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 3 datasets matching 'blight property violations'

1. **City of Pittsburgh Property Tax Delinquency**
   ID: `city-of-pittsburgh-property-tax-delinquency`
   This dataset contains all property in the City of Pittsburgh that is delinquent on property taxes for the given year.
   - Tax Delinquency (CSV) [DataStore] ID: `ed0d1550-c300-4114-865c-82dc7c23235b`
   - Tax Delinquency Data Dictionary with Definitions (XLSX) [DataStore] ID: `3508a696-2680-447b-8e45-64d485f52665`

2. **Pittsburgh PLI/DOMI/ES Violations Report**
   ID: `pittsburgh-pli-violations-report`
   **UPDATE: In March of 2026,
```


In [ ]:
# Step 2: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'blight property violations', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Load Data from Resource

**Resource ID:** `044f2016-1dfd-4ab0-bc1e-065da05fca2e`
**Limit:** 5

**Result preview:**
```
Resource: 044f2016-1dfd-4ab0-bc1e-065da05fca2e
Total records: 340,996
Loaded: 5
Fields (15): PK, CCR, HIERARCHY, INCIDENTTIME, INCIDENTLOCATION, CLEAREDFLAG, INCIDENTNEIGHBORHOOD, INCIDENTZONE, INCIDENTHIERARCHYDESC, OFFENSES, INCIDENTTRACT, COUNCIL_DISTRICT, PUBLIC_WORKS_DIVISION, X, Y

Sample (5 rows):

     PK      CCR  HIERARCHY        INCIDENTTIME                         INCIDENTLOCATION CLEAREDFLAG        INCIDENTNEIGHBORHOOD INCIDENTZONE                    INCIDENTHIERARCHYDESC                                 OFFENSES INCIDENTTRACT COUNCIL_DISTRICT PUBLIC_WORKS_DIVISION          X      
```


In [ ]:
# Step 3: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '044f2016-1dfd-4ab0-bc1e-065da05fca2e', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 4: Load Data from Resource

**Resource ID:** `70c06278-92c5-4040-ab28-17671866f81c`
**Limit:** 5

**Result preview:**
```
Resource: 70c06278-92c5-4040-ab28-17671866f81c
Total records: 572,222
Loaded: 5
Fields (22): casefile_number, address, parcel_id, status, case_file_type, investigation_date, investigation_outcome, investigation_findings, violation_description, violation_code_section, violation_code_section_title, violation_code_section_text, violation_spec_instructions, court_date, court_time, docket_number, court_decision, latitude, longitude, neighborhood, ward, council_district

Sample (5 rows):

   casefile_number                                address        parcel_id status  case_file_type investigation_
```


In [ ]:
# Step 4: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '70c06278-92c5-4040-ab28-17671866f81c', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 5: SQL Analysis Query

**SQL:**
```sql
SELECT "INCIDENTNEIGHBORHOOD", COUNT(*) as crime_count
FROM "044f2016-1dfd-4ab0-bc1e-065da05fca2e"
WHERE "INCIDENTNEIGHBORHOOD" IS NOT NULL AND "INCIDENTNEIGHBORHOOD" != ''
GROUP BY "INCIDENTNEIGHBORHOOD"
ORDER BY crime_count DESC
LIMIT 30
```

**Result preview:**
```
SQL: SELECT "INCIDENTNEIGHBORHOOD", COUNT(*) as crime_count
FROM "044f2016-1dfd-4ab0-bc1e-065da05fca2e"
WHERE "INCIDENTNEIGHBORHOOD" IS NOT NULL AND "INCIDENTNEIGHBORHOOD" != ''
GROUP BY "INCIDENTNEIGHBORHOOD"
ORDER BY crime_count DESC
LIMIT 30
Rows: 30
Columns: INCIDENTNEIGHBORHOOD, crime_count

       INCIDENTNEIGHBORHOOD  crime_count
  Central Business District        19943
           South Side Flats        19098
                    Carrick        11927
                  Shadyside         8773
                 Bloomfield         8704
               East Liberty         8451
        Squirre
```


In [ ]:
# Step 5: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "INCIDENTNEIGHBORHOOD", COUNT(*) as crime_count\nFROM "044f2016-1dfd-4ab0-bc1e-065da05fca2e"\nWHERE "INCIDENTNEIGHBORHOOD" IS NOT NULL AND "INCIDENTNEIGHBORHOOD" != \'\'\nGROUP BY "INCIDENTNEIGHBORHOOD"\nORDER BY crime_count DESC\nLIMIT 30'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 6: SQL Analysis Query

**SQL:**
```sql
SELECT "neighborhood", COUNT(DISTINCT "casefile_number") as blight_count
FROM "70c06278-92c5-4040-ab28-17671866f81c"
WHERE "neighborhood" IS NOT NULL AND "neighborhood" != ''
GROUP BY "neighborhood"
ORDER BY blight_count DESC
LIMIT 30
```

**Result preview:**
```
SQL: SELECT "neighborhood", COUNT(DISTINCT "casefile_number") as blight_count
FROM "70c06278-92c5-4040-ab28-17671866f81c"
WHERE "neighborhood" IS NOT NULL AND "neighborhood" != ''
GROUP BY "neighborhood"
ORDER BY blight_count DESC
LIMIT 30
Rows: 30
Columns: neighborhood, blight_count

             neighborhood  blight_count
         South Side Flats         11708
        South Side Slopes         10529
                  Carrick          5863
                Brookline          5392
         Mount Washington          5360
                Hazelwood          4894
      Squirrel Hill South         
```


In [ ]:
# Step 6: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "neighborhood", COUNT(DISTINCT "casefile_number") as blight_count\nFROM "70c06278-92c5-4040-ab28-17671866f81c"\nWHERE "neighborhood" IS NOT NULL AND "neighborhood" != \'\'\nGROUP BY "neighborhood"\nORDER BY blight_count DESC\nLIMIT 30'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **I now have all the data I need. Let me cross-reference these two lists to find the neighborhoods that appear in both top rankings.

---

## 🏘️ Pittsburgh Neighborhoods with High Crime AND High Blight

By cross-referencing the **Police Incident Blotter (Archived)** (340,996 crime incidents) and the **Pittsburgh PLI/DOMI/ES Violations Report** (572,222+ blight cases), here are the neighborhoods that rank highly on **both** lists:

---

### 🔴 Top Dual-Risk Neighborhoods

| Neighborhood | Crime Count | Crime Rank | Blight Cases | Blight Rank |
|---|---|---|---|---|
| **South Side Flats** | 19,098 | #2 | 11,708 | #1 |
| **Carrick** | 11,927 | #3 | 5,863 | #3 |
| **Bloomfield** | 8,704 | #5 | 3,947 | #11 |
| **Brookline** | 7,792 | #8 | 5,392 | #4 |
| **Mount Washington** | 7,723 | #9 | 5,360 | #5 |
| **Homewood South** | 7,297 | #10 | 2,268 | #26 |
| **Lincoln-Lemington-Belmar** | 6,865 | #11 | 2,387 | #25 |
| **Knoxville** | 6,435 | #13 | 3,590 | #14 |
| **Sheraden** | 6,214 | #15 | 3,686 | #12 |
| **Marshall-Shadeland** | 5,693 | #18 | 3,456 | #15 |
| **Hazelwood** | 5,543 | #19 | 4,894 | #6 |
| **Beechview** | 5,764 | #17 | 4,202 | #8 |
| **Homewood North** | 6,597 | #12 | 3,113 | #16 |
| **Allentown** | 5,273 | #20 | 2,625 | #20 |
| **Brighton Heights** | 6,305 | #14 | 2,579 | #21 |
| **Central Lawrenceville** | 4,385 | #26 | 2,494 | #24 |

---

### 🔑 Key Takeaways

1. 🥇 **South Side Flats** stands out as the most severe case — **#2 in crime** and **#1 in blight** across all Pittsburgh neighborhoods.
2. 🏚️ **Carrick, Brookline, and Mount Washington** form a tight cluster of high concern in both categories.
3. **Hazelwood and Beechview** show disproportionately high blight relative to crime, suggesting physical decay may be outpacing policing burdens there.
4. **Homewood South, Lincoln-Lemington-Belmar, and Homewood North** lean more heavily crime-weighted, which may reflect social instability alongside physical blight.

---

### ⚠️ Caveats
- **Crime data** comes from archived blotter records (through Nov. 2023); a new RMS system is in use for more recent data.
- **Blight cases** count distinct case file numbers from the PLI violations system — a property can have multiple violations per case.
- High counts in some neighborhoods (e.g., Central Business District) may partly reflect higher density, not just higher disorder.**

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-04-21

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-04-21 15:33:00
- **Query**: Which neighborhoods have both high crime and high blight?
- **Data Source**: WPRDC

---

### 💬 Need Help?

- **Documentation**: [Data Concierge Guide](https://github.com/dathere/data-concierge)
- **Issues**: Report problems or request features on GitHub
- **Extend**: Add your own analysis cells below this one!

---

*Generated by  AI Data Concierge v0.1.0*
